# Day 2 — Fine-tune from the Day 1 checkpoint (Kaggle T4)

**This is the v2 plan.** From-scratch training was overfitting (val PSNR 17 dB at epoch 34, lower than the 22.77 dB you already have). Instead we **fine-tune** the existing `final.pth` from your Day 1 dataset on LOL-v2 Real for 50 epochs, twice:

- **Run A — `baseline_ft`**: fine-tune **without** the illumination prior. Adapts the existing model to LOL-v2 Real. ~1.5 h on T4.
- **Run B — `illum_ft`**: fine-tune **with** the illumination prior (use_illum_prior=1). New 7th input channel of the head conv is zero-initialized so day-0 behavior matches the original model. ~1.5 h on T4.

A vs B is the **method ablation row** for the paper — same starting weights, same data, same epochs, only the architecture flag differs.

Lower learning rate (1e-5 instead of 1e-4) prevents the loss-climbing instability you saw before. Lower SSIM weight (0.3 instead of 0.5) for the same reason.

Total Kaggle GPU time: **~3.5 hours** (smoke test + 2 fine-tunes). Fits one session.

**Before running:** push your latest local commits (especially `--init-from`, `--lr` flags in `train.py`, and `kaggle_path_discovery.py`).

In [ ]:
# === Cell 1: install ===
!pip install -q --upgrade pip
!pip install -q scikit-image lpips matplotlib

In [ ]:
# === Cell 2: clone repo ===
import os, sys, shutil, subprocess

BRANCH = "main"
REPO_URL = "https://github.com/chirana07/Diffusion_new_final.git"
REPO_DIR = "/kaggle/working/Diffunet"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Verify the new flags exist on GitHub
with open("train.py") as f:
    train_src = f.read()
for flag in ("--init-from", "--lr", "--dataset-root"):
    if flag not in train_src:
        raise SystemExit(f"train.py on GitHub is missing {flag}. Push your latest commits first.")
print("All required flags present in train.py.")

if not os.path.exists("kaggle_path_discovery.py"):
    raise SystemExit("kaggle_path_discovery.py missing in repo. Push it first.")
print("kaggle_path_discovery.py present.")

In [ ]:
# === Cell 3: PREFLIGHT — discover datasets, set up canonical layout ===
# This cell does ZERO GPU work. Run it as many times as needed to debug paths.
import kaggle_path_discovery as kpd

kpd.diagnose("/kaggle/input")
discoveries = kpd.discover_all("/kaggle/input")
kpd.print_picks(discoveries)

# === MANUAL OVERRIDE if discovery picked the wrong directory ===
# Uncomment and edit the lines below if needed:
#
# discoveries['lolv2_real_train'] = {
#     'low_dir':  '/kaggle/input/<your-dataset>/path/to/Train/Low',
#     'high_dir': '/kaggle/input/<your-dataset>/path/to/Train/Normal',
#     'n_images': 689, 'kind': 'train'}
# discoveries['lolv2_real_test'] = {
#     'low_dir':  '/kaggle/input/<your-dataset>/path/to/Test/Low',
#     'high_dir': '/kaggle/input/<your-dataset>/path/to/Test/Normal',
#     'n_images': 100, 'kind': 'test'}

# Hard requirements: BOTH train and test for LOL-v2 Real must be found
if not discoveries.get('lolv2_real_train'):
    raise SystemExit(
        "\nLOL-v2 Real TRAIN split not found. The training script needs both Train and Test.\n"
        "Check the diagnose output above and override discoveries['lolv2_real_train'] in this cell.\n"
        "Most common cause: you uploaded only the Test split. Re-upload with the Train folder included."
    )
if not discoveries.get('lolv2_real_test'):
    raise SystemExit(
        "\nLOL-v2 Real TEST split not found. Re-attach the dataset via right panel + Add Data."
    )

# Symlink into canonical /kaggle/working/data/Real_captured/{Train,Test}/{Low,Normal}
DATASET_ROOT = kpd.setup_canonical_symlinks(discoveries, working_root="/kaggle/working")
print(f"\nCanonical layout assembled at: {DATASET_ROOT}")

# Validate
counts = kpd.validate_canonical(
    DATASET_ROOT,
    require_train=True, require_test=True, require_eval15=False,
    min_train_images=300, min_test_images=15,
)
print("Validated counts:")
for k, v in counts.items():
    print(f"  {k:<28s}  {v}")

In [ ]:
# === Cell 4: locate the Day 1 checkpoint (the one we fine-tune FROM) ===
import glob

ckpt_hits = sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True))
# Prefer files named final/best/last but containing 'pth_only' or no 'lolv2real' in name
# (we don't want to accidentally pick a previous Day 2 attempt's checkpoint)
preferred = [c for c in ckpt_hits if any(k in c.lower() for k in ('final',))]
if not preferred:
    preferred = [c for c in ckpt_hits if any(k in c.lower() for k in ('best', 'last'))]
preferred = [c for c in preferred if 'lolv2real' not in c.lower()]
INIT_FROM = preferred[0] if preferred else (ckpt_hits[0] if ckpt_hits else None)

# === MANUAL OVERRIDE if auto-pick is wrong ===
# INIT_FROM = "/kaggle/input/lumidiff-checkpoint/final.pth"

if INIT_FROM is None:
    raise SystemExit("No .pth checkpoint found under /kaggle/input/. Attach your Day 1 checkpoint.")
print(f"Will fine-tune from: {INIT_FROM}")
print(f"Size: {os.path.getsize(INIT_FROM) / 1e6:.1f} MB")

In [ ]:
# === Cell 5: GPU sanity ===
import torch
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - set Accelerator -> GPU T4")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 -> Save, then re-run.")

In [ ]:
# === Cell 6: SMOKE TEST — 1 epoch with --init-from and --use-illum-prior 1 ===
# This catches: bad checkpoint loading, dataset path issues, GPU OOM, layer shape mismatches.
# Cost: ~2-3 minutes. ALWAYS run this before kicking off the long fine-tunes.
import sys, subprocess
cmd = [
    sys.executable, "train.py",
    "--layout", "lolv2_real",
    "--dataset-root", DATASET_ROOT,
    "--init-from", INIT_FROM,
    "--use-illum-prior", "1",
    "--epochs", "1",
    "--crop-size", "256",
    "--batch-size", "4",
    "--lr", "1e-5",
    "--w-ssim", "0.3",
    "--num-workers", "2",
    "--val-max-batches", "5",
    "--tag", "smoke",
]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)
print("\nSmoke test passed. Proceed to Cell 7.")

In [ ]:
# === Cell 7: fine-tune training config (edit if needed) ===
# These hyperparameters are deliberately conservative for fine-tuning.
FT_EPOCHS    = 50    # 30-60 reasonable; 50 is the sweet spot
FT_LR        = "1e-5"  # 10x lower than from-scratch — essential for fine-tuning stability
FT_WSSIM     = "0.3"  # was 0.5; lower to avoid the loss-climbing you saw
FT_BATCH     = 4
FT_CROP      = 256
FT_VAL_EVERY = 5

os.environ["VAL_EVERY"] = str(FT_VAL_EVERY)

print("Fine-tune config:")
print(f"  epochs   = {FT_EPOCHS}")
print(f"  lr       = {FT_LR}")
print(f"  w_ssim   = {FT_WSSIM}")
print(f"  batch    = {FT_BATCH}")
print(f"  crop     = {FT_CROP}")
print(f"  val_every= {FT_VAL_EVERY}")

In [ ]:
# === Cell 8: RUN A — fine-tune WITHOUT illum prior (baseline_ft) ===
# Output: ./checkpoints/best_baseline_ft.pth + last_baseline_ft.pth
# Wall time: ~1.5 hours on T4.
cmd = [
    sys.executable, "train.py",
    "--layout", "lolv2_real",
    "--dataset-root", DATASET_ROOT,
    "--init-from", INIT_FROM,
    "--use-illum-prior", "0",
    "--epochs", str(FT_EPOCHS),
    "--crop-size", str(FT_CROP),
    "--batch-size", str(FT_BATCH),
    "--lr", FT_LR,
    "--w-ssim", FT_WSSIM,
    "--num-workers", "2",
    "--tag", "baseline_ft",
]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# === Cell 9: RUN B — fine-tune WITH illum prior (illum_ft) — THE HEADLINE ===
# Output: ./checkpoints/best_illum_ft.pth + last_illum_ft.pth
# Wall time: ~1.5 hours on T4.
cmd = [
    sys.executable, "train.py",
    "--layout", "lolv2_real",
    "--dataset-root", DATASET_ROOT,
    "--init-from", INIT_FROM,
    "--use-illum-prior", "1",
    "--epochs", str(FT_EPOCHS),
    "--crop-size", str(FT_CROP),
    "--batch-size", str(FT_BATCH),
    "--lr", FT_LR,
    "--w-ssim", FT_WSSIM,
    "--num-workers", "2",
    "--tag", "illum_ft",
]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# === Cell 10: collect both checkpoints + train logs into /kaggle/working ===
import shutil, glob, csv
OUT = "/kaggle/working/day2_checkpoints"
os.makedirs(OUT, exist_ok=True)

for src in glob.glob("./checkpoints/best_baseline_ft.pth") + \
           glob.glob("./checkpoints/best_illum_ft.pth") + \
           glob.glob("./checkpoints/last_baseline_ft.pth") + \
           glob.glob("./checkpoints/last_illum_ft.pth") + \
           glob.glob("./checkpoints/train_log_baseline_ft.csv") + \
           glob.glob("./checkpoints/train_log_illum_ft.csv"):
    shutil.copy2(src, OUT)
    print("copied", src)

# Print the val_psnr trajectory so you can sanity-check convergence
for log in sorted(glob.glob(os.path.join(OUT, "train_log_*.csv"))):
    print(f"\n--- {os.path.basename(log)} ---")
    rows = list(csv.reader(open(log)))
    header = rows[0]
    val_psnr_idx = header.index("val_psnr")
    val_ssim_idx = header.index("val_ssim")
    print("epoch  val_psnr  val_ssim")
    for r in rows[1:]:
        if r[val_psnr_idx] not in ("-1.0", ""):
            print(f"{r[0]:<5}  {float(r[val_psnr_idx]):>7.3f}  {float(r[val_ssim_idx]):.4f}")

out_zip = "/kaggle/working/day2_checkpoints.zip"
if os.path.exists(out_zip):
    os.remove(out_zip)
shutil.make_archive(out_zip.replace(".zip", ""), "zip", OUT)
print("\nWrote", out_zip)
print("\nNext: Output tab -> download day2_checkpoints.zip")
print("Then: Datasets -> + New Dataset, upload the .pth files, name it 'lumidiff-day2-ckpts'.")
print("Then: open kaggle_eval_illum.ipynb, attach all three datasets, Run All.")